# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genius1oak/ConvexML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb huggingface_hub

import os
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

One row = one content item for one client for one report date,
covering the March 2026 reporting month.

Source table: fact_content_daily_performance


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature
- `gsc_impressions` — historical search impressions used to measure performance.
- `gsc_clicks` — historical search clicks.
- `gsc_avg_position` — historical average search position.
- `visible_queries` — number of visible queries associated with the content item.
- `rare_impressions_share` — share of impressions from rare queries.
- `anonymized_impressions_share` — share of impressions from anonymized queries.
- `top_query_share` — share of retained impressions coming from the top query.

### Label
- `is_declining` — whether impressions decline by more than 20% compared with the previous period.

### Context
- `client_hash_id` — identifies the client.
- `content_hash_id` — identifies the content item.
- `report_date` — identifies the reporting date/month.

### Excluded
- `ga4_*` fields — excluded because this Search Intelligence lane focuses on Google Search Console performance rather than Analytics.
- `client_hash_id` and `content_hash_id` are used for grouping and identification, not as predictive features, because the model should learn from content/search behavior rather than pseudonymous IDs.
- Future-period performance fields — excluded from features because they would cause data leakage.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# Query 1 — verify the grain for March 2026

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS distinct_grain_rows,
        COUNT(*) - COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS duplicate_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,duplicate_rows
0,9841378,9841378,0


In [18]:
# Query 2 — verify March 2026 row count and date span

march_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

march_check

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [19]:
# Query 3 — verify GSC availability using IS TRUE

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows
0,9841378,3611061


### Five features

| Feature | What it measures | Available when? |
|---|---|---|
| `imp_prev30` | Impressions during the previous 30-day period | Available before the prediction window because it uses only completed historical days. |
| `pos_volatility` | Variation in search position during the historical period | Available before the prediction window because it is calculated from historical GSC observations. |
| `rare_share` | Share of retained impressions from rare queries | Available before the prediction window because it comes from historical query data. |
| `anon_share` | Share of retained impressions from anonymized queries | Available before the prediction window because it comes from historical query data. |
| `top_query_share` | Share of retained impressions contributed by the strongest query | Available before the prediction window because it comes from historical query data. |

In [20]:
# Build the five-feature frame

feature_frame = con.sql(f"""
    WITH daily_features AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS imp_prev30,

            STDDEV_SAMP(gsc_avg_position) AS pos_volatility

        FROM {TABLES['fact_daily']}

        WHERE report_date >= DATE '2026-01-30'
          AND report_date < DATE '2026-03-01'
          AND gsc_data_available IS TRUE

        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
    ),

    query_features AS (
        SELECT
            content_hash_id,

            ANY_VALUE(rare_impressions_share) AS rare_share,
            ANY_VALUE(anonymized_impressions_share) AS anon_share,

            MAX(impressions_90d) / NULLIF(SUM(impressions_90d), 0)
                AS top_query_share

        FROM {TABLES['fact_query_90d']}

        GROUP BY content_hash_id
    )

    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.imp_prev30,
        d.pos_volatility,
        q.rare_share,
        q.anon_share,
        q.top_query_share

    FROM daily_features d
    LEFT JOIN query_features q
        ON d.content_hash_id = q.content_hash_id
""").df()

print(f"Feature rows: {len(feature_frame):,}")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 81,521


,client_hash_id,content_hash_id,imp_prev30,pos_volatility,rare_share,anon_share,top_query_share
0,client_08a6a72ff48e62c0,content_44e29d7baeb1cab0,2611.0,1.622027,0.132581,0.768887,0.097192
1,client_08a6a72ff48e62c0,content_4514f9e55a925af3,131.0,27.516019,0.204082,0.711370,0.448276
2,client_08a6a72ff48e62c0,content_453d3d950a06589a,187.0,7.024905,0.063402,0.030171,0.128799
3,client_08a6a72ff48e62c0,content_458d63408989b5e6,4452.0,1.088739,0.018495,0.764399,0.166898
4,client_08a6a72ff48e62c0,content_459bf6e509b4e372,141.0,12.294647,0.131148,0.707260,0.275362


In [21]:
# Deliberate leakage experiment
# This feature is created from the outcome itself, so it should NOT be used.

leak_test = feature_frame.copy()

# March outcome: impressions declined by more than 20% vs the previous 30 days
march_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

leak_test = leak_test.merge(
    march_outcome,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

leak_test['is_declining'] = (
    leak_test['imp_march'] < 0.8 * leak_test['imp_prev30']
).astype(int)

# Intentionally leaked feature: it directly uses the March outcome.
leak_test['leaked_label'] = leak_test['is_declining']

print(
    "Correlation between leaked feature and label:",
    leak_test['leaked_label'].corr(leak_test['is_declining'])
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Correlation between leaked feature and label: 0.9999999999999999


### Leakage trap

I deliberately created `leaked_label` using the outcome I was trying to predict. The correlation with the true label was **1.0**, demonstrating perfect leakage.

This feature is not available at the decision moment because it uses the future outcome. It was therefore removed from the honest feature set.

The lesson is that a very strong model result is not necessarily useful if the model has access to information that would only become known after the prediction period.

In [22]:
# Remove the deliberately leaked field and keep only the five honest features

honest_features = [
    'imp_prev30',
    'pos_volatility',
    'rare_share',
    'anon_share',
    'top_query_share'
]

final_feature_frame = feature_frame[honest_features].copy()

print("Final honest features:")
print(final_feature_frame.columns.tolist())

Final honest features:
['imp_prev30', 'pos_volatility', 'rare_share', 'anon_share', 'top_query_share']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- **Unbalanced history:** clients do not all have the same amount of historical GSC data, so the available feature windows differ across clients.
- **GSC-only history:** some clients have GSC data without GA4 data, so this slice cannot describe downstream analytics outcomes for those clients.
- **Window overlaps:** rolling time windows can overlap across observations, so nearby observations may not be fully independent.
- **Search-only view:** GSC data can show search visibility, clicks, impressions and position, but it cannot by itself explain why a page gained or lost visibility.

In [23]:
# Check the range of GSC history available across clients

history_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        COUNT(gsc_data_start) AS clients_with_gsc_start,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start
    FROM {TABLES['dim_clients']}
""").df()

history_check

,total_clients,clients_with_gsc_start,earliest_gsc_start,latest_gsc_start
0,104,67,2025-01-27,2026-06-02


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.